In [1]:
"""
train_efficientnet_hf.py
========================
Train EfficientNet-B0 on barcode_dataset_hf_final.

Dataset expected:
  barcode_dataset_hf_final/
    train/
      PASS/
      FAIL/
    validation/
      PASS/
      FAIL/
    test/
      PASS/
      FAIL/

Run:
  pip install torch torchvision scikit-learn matplotlib seaborn opencv-python
  python train_efficientnet_hf.py
"""

import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# Config
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATASET_DIR = "barcode_dataset_hf_final"
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE_HEAD = 1e-4
LEARNING_RATE_FULL = 1e-5
NUM_WORKERS = 0          # keep 0 on Windows
MODEL_SAVE = "best_barcode_model_efficientnet_hf.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# Prints
# ============================================================

print("=" * 70)
print("Barcode Damage Detection — EfficientNet-B0 Training")
print("=" * 70)
print(f"Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Dataset       : {DATASET_DIR}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Model save    : {MODEL_SAVE}")

# ============================================================
# Transforms
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# ============================================================
# Dataset + Loaders
# ============================================================

train_ds = datasets.ImageFolder(
    os.path.join(DATASET_DIR, "train"),
    transform=train_transform
)
val_ds = datasets.ImageFolder(
    os.path.join(DATASET_DIR, "validation"),
    transform=eval_transform
)
test_ds = datasets.ImageFolder(
    os.path.join(DATASET_DIR, "test"),
    transform=eval_transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

CLASS_NAMES = train_ds.classes

print("\n" + "=" * 70)
print("Dataset Info")
print("=" * 70)
print(f"Classes       : {CLASS_NAMES}")
print(f"Train images  : {len(train_ds)}")
print(f"Val images    : {len(val_ds)}")
print(f"Test images   : {len(test_ds)}")

# ============================================================
# Model
# ============================================================

print("\n" + "=" * 70)
print("Loading EfficientNet-B0")
print("=" * 70)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze backbone first
for param in model.parameters():
    param.requires_grad = False

# Replace final classifier
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(CLASS_NAMES))

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[1].parameters(), lr=LEARNING_RATE_HEAD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Training strategy:")
print("- epochs 1-5   : train classifier head only")
print("- epochs 6-end : fine-tune full network")

# ============================================================
# Training loop
# ============================================================

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 50)

    # Unfreeze after 5 epochs
    if epoch == 5:
        print("Unfreezing full backbone for fine-tuning...")
        for param in model.parameters():
            param.requires_grad = True

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE_FULL)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2
        )

    # ---------------- Training ----------------
    model.train()
    running_loss = 0.0
    running_correct = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()

        if (batch_idx + 1) % 20 == 0:
            print(f"Batch {batch_idx + 1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_ds)
    train_acc = running_correct / len(train_ds)

    # ---------------- Validation ----------------
    model.eval()
    val_running_loss = 0.0
    val_running_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            val_running_loss += loss.item() * images.size(0)
            val_running_correct += (preds == labels).sum().item()

    val_loss = val_running_loss / len(val_ds)
    val_acc = val_running_correct / len(val_ds)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc * 100:.2f}%")
    print(f"LR: {current_lr:.6f} | Time: {time.time() - t0:.1f}s")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, MODEL_SAVE)
        print(f"New best model saved with val acc = {best_val_acc * 100:.2f}%")

print("\n" + "=" * 70)
print("Training Complete")
print("=" * 70)
print(f"Best validation accuracy: {best_val_acc * 100:.2f}%")
print(f"Best model saved to     : {MODEL_SAVE}")

# ============================================================
# Save training curves
# ============================================================

plt.figure(figsize=(12, 5))
plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_loss_efficientnet_hf.png", dpi=150)
plt.close()

plt.figure(figsize=(12, 5))
plt.plot([x * 100 for x in history["train_acc"]], label="Train Accuracy", linewidth=2)
plt.plot([x * 100 for x in history["val_acc"]], label="Val Accuracy", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy over Epochs")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_accuracy_efficientnet_hf.png", dpi=150)
plt.close()

print("Saved:")
print("- training_loss_efficientnet_hf.png")
print("- training_accuracy_efficientnet_hf.png")

# ============================================================
# Test evaluation
# ============================================================

print("\n" + "=" * 70)
print("Evaluating on Test Set")
print("=" * 70)

model.load_state_dict(torch.load(MODEL_SAVE, map_location=DEVICE))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"Accuracy  : {acc * 100:.2f}%")
print(f"Precision : {prec * 100:.2f}%")
print(f"Recall    : {rec * 100:.2f}%")
print(f"F1-score  : {f1 * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title("Confusion Matrix - EfficientNet-B0")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix_efficientnet_hf.png", dpi=150)
plt.close()

print("Saved:")
print("- confusion_matrix_efficientnet_hf.png")

Barcode Damage Detection — EfficientNet-B0 Training
Device        : cpu
Dataset       : barcode_dataset_hf_final
Batch size    : 32
Epochs        : 15
Model save    : best_barcode_model_efficientnet_hf.pth

Dataset Info
Classes       : ['FAIL', 'PASS']
Train images  : 4200
Val images    : 900
Test images   : 900

Loading EfficientNet-B0
Training strategy:
- epochs 1-5   : train classifier head only
- epochs 6-end : fine-tune full network

Epoch 1/15
--------------------------------------------------
Batch 20/132 | Loss: 0.7190
Batch 40/132 | Loss: 0.7021
Batch 60/132 | Loss: 0.6181
Batch 80/132 | Loss: 0.6374
Batch 100/132 | Loss: 0.6267
Batch 120/132 | Loss: 0.5780
Train Loss: 0.6433 | Train Acc: 65.38%
Val   Loss: 0.5936 | Val   Acc: 73.22%
LR: 0.000100 | Time: 218.9s
New best model saved with val acc = 73.22%

Epoch 2/15
--------------------------------------------------
Batch 20/132 | Loss: 0.5190
Batch 40/132 | Loss: 0.5613
Batch 60/132 | Loss: 0.5927
Batch 80/132 | Loss: 0.5748
B

In [2]:
"""
Barcode Damage Detection — Improved Live Webcam Inference
==========================================================
Improvements over original:
  1. Faster — rotation only tested once every 30 frames, not every frame
  2. Smoother — result is stabilised over last 10 frames (no flickering)
  3. Smarter detection — aspect ratio filter removes false barcode detections
  4. Visual — confidence bar shows how sure the model is
  5. FPS counter displayed live so you can see performance
  6. Freeze frame — press F to freeze/unfreeze the current result
  7. Snapshot — press S to save the current frame as a JPEG

Run:
    python webcam_inference.py

Requires best_barcode_model.pth (or best_barcode_model_efficientnet.pth)
in the same folder.
"""

import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, models
from collections import deque
import time
import os

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_PATH      = "best_barcode_model_efficientnet_hf.pth"   # change if using ResNet
MODEL_TYPE      = "efficientnet"   # "efficientnet" or "resnet"
CLASS_NAMES     = ["FAIL", "PASS"]
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SMOOTHING_FRAMES    = 10   # average predictions over this many frames
ROTATION_CHECK_FREQ = 30   # re-check best rotation every N frames
SNAPSHOT_DIR        = "snapshots"
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

# ── Load model ────────────────────────────────────────────────────────────────
print(f"Loading model from {MODEL_PATH} ...")

if MODEL_TYPE == "efficientnet":
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
else:  # resnet
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(f"  Model loaded on {DEVICE}")

# ── Transforms ────────────────────────────────────────────────────────────────
infer_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# ── Barcode region detection ──────────────────────────────────────────────────

def detect_barcode_region(frame):
    """
    Locate the barcode in the frame using vertical edge density.

    Improvements over original:
      - Aspect ratio filter: real barcodes are wider than tall (ratio > 1.2)
        This removes false detections on other rectangular objects
      - Scores each contour by edge density inside it, not just area
        A large low-density region (like a window) is rejected
    """
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w  = gray.shape
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray  = clahe.apply(gray)

    sobel  = np.uint8(np.absolute(cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)))
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    closed = cv2.morphologyEx(sobel, cv2.MORPH_CLOSE, kernel)
    _, thresh = cv2.threshold(closed, 50, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    best_score = 0
    best_box   = None

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)

        # Minimum size: at least 1% of frame
        if cw * ch < w * h * 0.01:
            continue

        # Aspect ratio: barcodes are wider than tall
        # Skip very square or very tall regions
        aspect = cw / max(ch, 1)
        if aspect < 0.8:
            continue

        # Edge density score: sum of sobel values inside the bounding box
        roi_sobel = sobel[y:y+ch, x:x+cw]
        density   = roi_sobel.mean()

        # Prefer high edge density + reasonably sized regions
        score = density * (cw * ch) / (w * h)
        if score > best_score:
            best_score = score
            best_box   = (x, y, cw, ch)

    if best_box is not None:
        x, y, cw, ch = best_box
        pad = 20
        x1  = max(0, x - pad);    y1 = max(0, y - pad)
        x2  = min(w, x + cw + pad); y2 = min(h, y + ch + pad)
        return frame[y1:y2, x1:x2], (x1, y1, x2, y2), True

    # Fallback: centre 60%
    x1 = int(w * 0.20); x2 = int(w * 0.80)
    y1 = int(h * 0.20); y2 = int(h * 0.80)
    return frame[y1:y2, x1:x2], (x1, y1, x2, y2), False


# ── Single-rotation inference ─────────────────────────────────────────────────

def run_model(crop_rgb):
    """Run model on one RGB crop, return (pred_index, pass_confidence)."""
    tensor = infer_transform(crop_rgb).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out   = model(tensor)
        probs = torch.softmax(out, dim=1)[0]
    return out.argmax(dim=1).item(), probs[1].item()  # pred, pass_conf


def find_best_rotation(crop_bgr):
    """
    Try all 4 rotations, return the rotation index that gives
    highest PASS confidence. Called once every ROTATION_CHECK_FREQ frames.
    """
    rgb       = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    rotations = [
        rgb,
        cv2.rotate(rgb, cv2.ROTATE_90_CLOCKWISE),
        cv2.rotate(rgb, cv2.ROTATE_180),
        cv2.rotate(rgb, cv2.ROTATE_90_COUNTERCLOCKWISE),
    ]
    best_pass = -1
    best_rot  = 0
    for i, rot in enumerate(rotations):
        _, pass_conf = run_model(rot)
        if pass_conf > best_pass:
            best_pass = pass_conf
            best_rot  = i
    return best_rot


ROTATION_CODES = [None,
                  cv2.ROTATE_90_CLOCKWISE,
                  cv2.ROTATE_180,
                  cv2.ROTATE_90_COUNTERCLOCKWISE]

def classify_with_rotation(crop_bgr, rot_idx):
    """Run model using the pre-determined best rotation."""
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    if ROTATION_CODES[rot_idx] is not None:
        rgb = cv2.rotate(rgb, ROTATION_CODES[rot_idx])
    pred, pass_conf = run_model(rgb)
    return CLASS_NAMES[pred], pass_conf * 100


# ── Drawing helpers ───────────────────────────────────────────────────────────

def draw_confidence_bar(frame, confidence, label, x, y, bar_w=200, bar_h=16):
    """Draw a filled confidence bar below the result text."""
    color     = (0, 200, 0) if label == "PASS" else (0, 0, 220)
    fill      = int(bar_w * confidence / 100)
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h),
                  (60, 60, 60), -1)                          # background
    cv2.rectangle(frame, (x, y), (x + fill, y + bar_h),
                  color, -1)                                  # fill
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h),
                  (120, 120, 120), 1)                         # border
    cv2.putText(frame, f"{confidence:.0f}%",
                (x + bar_w + 8, y + bar_h - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)


def draw_overlay(frame, label, confidence, box, detected, fps, frozen):
    """Draw the full result overlay on the frame."""
    x1, y1, x2, y2 = box
    color     = (0, 200, 0) if label == "PASS" else (0, 0, 220)
    box_color = color if detected else (120, 120, 120)

    # Black header bar
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 90), (0, 0, 0), -1)

    # Main label
    cv2.putText(frame, label,
                (20, 58), cv2.FONT_HERSHEY_SIMPLEX, 2.0, color, 3)

    # Confidence bar next to label
    draw_confidence_bar(frame, confidence, label, 200, 30)

    # FPS top right
    fps_text = f"FPS: {fps:.0f}"
    if frozen:
        fps_text = "FROZEN — press F to resume"
    cv2.putText(frame, fps_text,
                (frame.shape[1] - 260, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                (180, 180, 180), 1)

    # Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

    # Corner accents on the box (looks cleaner than a full rectangle)
    corner_len = 20
    corners = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]
    for cx, cy in corners:
        dx = corner_len if cx == x1 else -corner_len
        dy = corner_len if cy == y1 else -corner_len
        cv2.line(frame, (cx, cy), (cx + dx, cy), color, 2)
        cv2.line(frame, (cx, cy), (cx, cy + dy), color, 2)

    # Tag below box
    tag = "Barcode detected" if detected else "No barcode — centre crop"
    cv2.putText(frame, tag,
                (x1, max(y1 - 10, 95)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, box_color, 1)

    # Controls reminder at bottom
    h = frame.shape[0]
    cv2.putText(frame, "Q = quit   F = freeze   S = snapshot",
                (12, h - 12), cv2.FONT_HERSHEY_SIMPLEX,
                0.45, (140, 140, 140), 1)

    return frame


# ── Main webcam loop ──────────────────────────────────────────────────────────

def run_webcam():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("  Webcam not found.")
        return

    # Set higher resolution if camera supports it
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    print("  Controls:")
    print("    Q — quit")
    print("    F — freeze / unfreeze current frame")
    print("    S — save snapshot to snapshots/ folder")

    # State
    smoothing_buffer = deque(maxlen=SMOOTHING_FRAMES)  # stores pass_conf values
    best_rotation    = 0       # updated every ROTATION_CHECK_FREQ frames
    frame_count      = 0
    frozen           = False
    frozen_frame     = None
    fps_buffer       = deque(maxlen=30)
    last_label       = "PASS"
    last_conf        = 0.0
    last_box         = (0, 0, 100, 100)
    last_detected    = False
    snapshot_count   = 0

    while True:
        t_start = time.time()

        if not frozen:
            ret, frame = cap.read()
            if not ret:
                break

            # ── Detect barcode region ─────────────────────────────────────────
            crop, box, detected = detect_barcode_region(frame)
            last_box      = box
            last_detected = detected

            # ── Re-check best rotation periodically ──────────────────────────
            if frame_count % ROTATION_CHECK_FREQ == 0:
                best_rotation = find_best_rotation(crop)

            # ── Classify with locked rotation ─────────────────────────────────
            label, pass_conf = classify_with_rotation(crop, best_rotation)
            smoothing_buffer.append(pass_conf)

            # ── Smooth result over last N frames ──────────────────────────────
            avg_pass_conf = np.mean(smoothing_buffer)
            if avg_pass_conf >= 50:
                last_label = "PASS"
                last_conf  = avg_pass_conf
            else:
                last_label = "FAIL"
                last_conf  = 100 - avg_pass_conf

            frame_count += 1

            # FPS
            elapsed = time.time() - t_start
            fps_buffer.append(1.0 / max(elapsed, 0.001))
            fps = np.mean(fps_buffer)

            # Draw overlay
            display = draw_overlay(frame.copy(), last_label, last_conf,
                                   last_box, last_detected, fps, frozen)
        else:
            display = frozen_frame.copy()
            fps = 0

        cv2.imshow("Barcode Damage Detector  |  Q=quit  F=freeze  S=snapshot",
                   display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("f"):
            frozen = not frozen
            if frozen:
                frozen_frame = display.copy()
                print(f"  Frozen — result: {last_label} ({last_conf:.1f}%)")
            else:
                print("  Resumed")
        elif key == ord("s"):
            snapshot_count += 1
            filename = os.path.join(SNAPSHOT_DIR,
                                    f"snapshot_{snapshot_count:03d}_{last_label}.jpg")
            cv2.imwrite(filename, display)
            print(f"  Snapshot saved: {filename}")

    cap.release()
    cv2.destroyAllWindows()
    print("  Webcam closed.")


# ── Standalone image prediction ───────────────────────────────────────────────

def predict_image(image_path):
    """
    Run inference on a single saved image file.
    Tries all 4 rotations and picks the highest PASS confidence.

    Usage:
        predict_image("my_barcode.jpg")
    """
    img  = cv2.imread(image_path)
    if img is None:
        print(f"  Could not read: {image_path}")
        return None, None
    crop, _, _ = detect_barcode_region(img)
    rot        = find_best_rotation(crop)
    label, conf = classify_with_rotation(crop, rot)
    print(f"  Image  : {image_path}")
    print(f"  Result : {label}  ({conf:.1f}% confidence)")
    return label, conf


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    run_webcam()

# To run on a single image instead:
# predict_image("my_barcode.jpg")

Loading model from best_barcode_model_efficientnet_hf.pth ...
  Model loaded on cpu
  Controls:
    Q — quit
    F — freeze / unfreeze current frame
    S — save snapshot to snapshots/ folder
  Frozen — result: PASS (67.9%)
  Resumed
  Frozen — result: PASS (92.9%)
  Resumed
  Frozen — result: FAIL (56.0%)
  Resumed
  Frozen — result: FAIL (96.7%)
  Resumed
  Frozen — result: FAIL (59.5%)
  Resumed
  Frozen — result: FAIL (86.0%)
  Resumed
  Frozen — result: PASS (90.0%)
  Resumed
  Frozen — result: FAIL (57.9%)
  Resumed
  Frozen — result: FAIL (61.8%)
  Resumed
  Frozen — result: FAIL (95.0%)
  Resumed
  Frozen — result: FAIL (95.3%)
  Resumed
  Frozen — result: FAIL (50.4%)
  Resumed
  Frozen — result: FAIL (54.7%)
  Resumed
  Frozen — result: PASS (93.0%)
  Resumed
  Frozen — result: PASS (93.1%)
  Resumed
  Frozen — result: FAIL (95.1%)
  Resumed
  Frozen — result: PASS (94.2%)
  Resumed
  Frozen — result: FAIL (57.2%)
  Resumed
  Frozen — result: FAIL (53.9%)
  Resumed
  Frozen —